# Hugging Face Login

In [ ]:
# GPU 메모리 세그먼트를 동적으로 확장 가능하게 만듬
# CUDA out of memory 라는 에러가 떠서 해당 코드 추가함
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
from huggingface_hub import login

# Hugging Face API 토큰을 입력합니다.
api_token = "..."
login(api_token)

In [ ]:
import torch
import wandb
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    BitsAndBytesConfig,   # ← 추가
    pipeline, 
    Trainer
)
from transformers.integrations import WandbCallback
from trl import DataCollatorForCompletionOnlyLM
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training  # ← 추가
import evaluate

# 모델을 허깅페이스에서 다운로드
model_name = "google/gemma-2b-it"

# ↓↓↓ 기존 from_pretrained 블록 전체를 아래로 교체 ↓↓↓
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# 토크나이저를 허깅페이스에서 다운로드(AutoModelForCausalLM)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,   # ← 4bit 양자화(메모리를 줄여서 로드, 너무 커서 이렇게 바꿈)
    device_map="auto", # 모델이 어떤 장치에서 실행될지 자동으로 결정(GPU가 있으면 GPU 선택, 없으면 CPU 선택)
    attn_implementation="eager", # 어텐션 연산은 eager 방식을 사용
)

model = prepare_model_for_kbit_training(model)  # ← gradient checkpointing 등 준비

# 이건 따로 추가함. 대형 언어 모델을 전체 학습하지 않고 일부 작은 파라미터만 학습하도록 만드는 설정
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # 학습 파라미터 수 확인용

tokenizer = AutoTokenizer.from_pretrained(model_name)

`config.hidden_act` is ignored, you should use `config.hidden_activation` instead.
Gemma's activation function will be set to `gelu_pytorch_tanh`. Please, use
`config.hidden_activation` if you want to override this behaviour.
See https://github.com/huggingface/transformers/pull/29402 for more details.
c:\Users\mobile\AppData\Local\mise\installs\python\3.12.13\Lib\site-packages\accelerate\utils\modeling.py:804: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  _ = torch.tensor([0], device=i)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 3,686,400 || all params: 2,509,858,816 || trainable%: 0.14687678751090355


### 3.4.3. 데이터셋 

In [ ]:
# 데이터셋 가져오기
import datasets
dataset = datasets.load_dataset("jaehy12/news3")
element = dataset["train"][1]
element

{'original': ' “동선 봐도 도움이 안 되는 것 같아요.\n  어딘지 알아야 피해가고 조심할 텐데요.\n ”“확진자가 다녀간 곳은 방역 마쳤는데 동선 공개는 그 부근 일대를 싹 다 죽이자는 것 같아요.\n ” 최근 부천시 페이스북 신종 코로나바이러스 감염증(코로나19) 관련 게시물에는 확진자 동선 공개 범위를 성토하는 댓글이 여럿 달렸다.\n  물류센터·교회 발 코로나19 확산이 지역사회 N차 감염으로 이어지면서 확진자 동선을 확대 공개해야 한다는 주장이 제기된 가운데 사생활 침해 등을 근거로 반대하는 의견도 나오면서 동선공개 논란에 다시 불이 붙는 모양새다.\n  지난 2월 코로나19 초기엔 확진자 동선이 대부분 공개됐다.\n  쇼핑몰을 방문한 확진자가 시간대별로 어느 매장을 찾았는지 등의 동선이 지자체 소셜네트워크서비스(SNS)에 게재됐다.\n  이후 사생활 침해 논란이 일자 질본은 감염병 예방에 필요한 정보에 한해 확진자 정보를 공개하라는 권고사항을 발표했다.\n  권고사항에는 증상 발생 2일 전부터 격리일까지의 시간, 감염을 우려할 만큼의 확진자와의 접촉이 일어난 장소 및 수단 등을 공개한다는 내용이 담겼다.\n  해당 공간 내 모든 접촉자가 파악되면 공개하지 않을 수 있다는 단서도 추가됐다.\n  확진자가 마지막 접촉자와 만날 일로부터 14일이 지나면 공개한 동선을 삭제할 것도 권고했다.\n  확진 늘자 동선공개 확대 나선 지자체 물류센터·교회 등 집단감염에 이어 감염경로가 특정되지 않은 확진 사례가 늘면서 일부 지자체는 동선 공개 범위를 확대하고 있다.\n  김포시는 지난 2일부터 확진자 이동 경로에 따른 동선 공개 원칙을 제한적으로 확대했다.\n  시민 우려 불식을 위해 확진자 방문으로 접촉자가 다수 발생하고 확산이 우려되는 장소는 상호를 공개하기로 한 것이다.\n  앞서 부천 118번 확진자인 A씨(31)가 한 제약회사 영업사원인 사실이 알려지면서 SNS에 그가 다닌 병원 정보가 공유됐다.\n  장덕천 부천시장은 “SNS에 A씨가 평소

### 3.4.4. Gemma 모델 기능 확인 

#### 키워드 추출 기능 확인

In [ ]:
input_text = """다음 텍스트를 한국어로 간단히 요약해주세요:\n부산의 한 왕복 2차선 도로에서 역주행 사고로 배달 오토바이 운전자인 고등학생이 숨지는 사고가 발생했다.
유족은 '가해자가 사고 후 곧바로 신고하지 않고 늑장 대응해 피해를 키웠다'고 주장하고 있다.
11일 부산진경찰서는 교통사고처리특례법(교통사고처리법)상 업무상 과실치사 혐의로 지난 3일 A(59)씨를 검찰에 불구속 송치했다고 밝혔다.
A씨는 교통사고처리법상 12대 중과실에 해당되는 '중앙선 침범'으로 역주행 교통사고를 일으킨 혐의를 받는다.
경찰에 따르면 스포츠유틸리티차량(SUV) 운전자 A씨는 5월 19일 밤 11시 50분쯤 부산진구 가야고가교 밑 도로에서 중앙선을 넘어 역주행으로 140m를 달려
반대편 차선의 오토바이 운전자 조모(16)군을 들이받았다. 조군은 원동기장치자전거 면허를 취득한 상태였고 헬멧도 쓰고 있었지만 크게 다쳤다.
사고 당일 수술을 받았으나 얼마 후 2차 뇌출혈로 뇌사 판정이 내려졌고, 사고 발생 약 한 달 만인 지난달 16일 끝내 사망했다.
사고를 낸 A씨는 술을 마시거나 약물을 복용한 상태에서 운전하지는 않은 것으로 조사됐다.
경찰 관계자는 'A씨가 자신이 정주행을 하고 오토바이가 역주행을 한 것으로 착각했다고 진술했다'고 설명했다."""

def change_inference_chat_format(input_text):
    return [
        {
            "role": "user", 
            "content": (
            "다음 텍스트에서 핵심 키워드 5개를 단어 형태로 추출해주세요:\n"
            "서울 강남구에서 음주운전 차량이 인도를 덮쳐 보행자 3명이 부상을 입었다."
            )
        },
        {"role": "assistant", "content": "음주운전, 강남구, 보행자, 인도 사고, 현행범"},
        {"role": "user", "content": f"다음 텍스트에서 핵심 키워드 5개를 단어 형태로 추출해주세요:\n{input_text}"},
        {"role": "assistant", "content": ""}
    ]
prompt = change_inference_chat_format(input_text)
# tokenizer 초기화 및 적용
inputs = tokenizer.apply_chat_template(
    prompt, 
    tokenize=True, 
    add_generation_prompt=True, 
    return_tensors="pt"
    ).to(model.device)
outputs = model.generate(
    input_ids=inputs.to(model.device), 
    max_new_tokens=256
    )
print(tokenizer.decode(
    outputs[0], 
    skip_special_tokens=True
    ))

user
다음 텍스트에서 핵심 키워드 5개를 단어 형태로 추출해주세요:
서울 강남구에서 음주운전 차량이 인도를 덮쳐 보행자 3명이 부상을 입었다.
model
음주운전, 강남구, 보행자, 인도 사고, 현행범
user
다음 텍스트에서 핵심 키워드 5개를 단어 형태로 추출해주세요:
다음 텍스트를 한국어로 간단히 요약해주세요:
부산의 한 왕복 2차선 도로에서 역주행 사고로 배달 오토바이 운전자인 고등학생이 숨지는 사고가 발생했다.
유족은 '가해자가 사고 후 곧바로 신고하지 않고 늑장 대응해 피해를 키웠다'고 주장하고 있다.
11일 부산진경찰서는 교통사고처리특례법(교통사고처리법)상 업무상 과실치사 혐의로 지난 3일 A(59)씨를 검찰에 불구속 송치했다고 밝혔다.
A씨는 교통사고처리법상 12대 중과실에 해당되는 '중앙선 침범'으로 역주행 교통사고를 일으킨 혐의를 받는다.
경찰에 따르면 스포츠유틸리티차량(SUV) 운전자 A씨는 5월 19일 밤 11시 50분쯤 부산진구 가야고가교 밑 도로에서 중앙선을 넘어 역주행으로 140m를 달려
반대편 차선의 오토바이 운전자 조모(16)군을 들이받았다. 조군은 원동기장치자전거 면허를 취득한 상태였고 헬멧도 쓰고 있었지만 크게 다쳤다.
사고 당일 수술을 받았으나 얼마 후 2차 뇌출혈로 뇌사 판정이 내려졌고, 사고 발생 약 한 달 만인 지난달 16일 끝내 사망했다.
사고를 낸 A씨는 술을 마시거나 약물을 복용한 상태에서 운전하지는 않은 것으로 조사됐다.
경찰 관계자는 'A씨가 자신이 정주행을 하고 오토바이가 역주행을 한 것으로 착각했다고 진술했다'고 설명했다.
model

model
- 음주운전
- 강남구
- 보행자
- 인도
- 교통사고처리법


키워드를 생성하는 능력이 Gemma에게 있는 것을 확인했습니다.

#### 요약 기능 확인

In [8]:
# Input_text : 위 정의된 기사와 동일 

def change_inference_chat_format(input_text):
    return [
    {"role": "user", "content": f"{input_text}"},
    {"role": "assistant", "content": "한국어 요약:\n"}
    ]

# chat template 적용
prompt = change_inference_chat_format(input_text)

# 생성
inputs = tokenizer.apply_chat_template(prompt, tokenize=True, add_generation_prompt=True, return_tensors="pt")
# 여기 수정
outputs = model.generate(
    inputs,
    max_new_tokens=256,
    use_cache=True
    )
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

c:\Users\mobile\AppData\Local\mise\installs\python\3.12.13\Lib\site-packages\transformers\generation\utils.py:1885: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(


user
다음 텍스트를 한국어로 간단히 요약해주세요:
부산의 한 왕복 2차선 도로에서 역주행 사고로 배달 오토바이 운전자인 고등학생이 숨지는 사고가 발생했다.
유족은 '가해자가 사고 후 곧바로 신고하지 않고 늑장 대응해 피해를 키웠다'고 주장하고 있다.
11일 부산진경찰서는 교통사고처리특례법(교통사고처리법)상 업무상 과실치사 혐의로 지난 3일 A(59)씨를 검찰에 불구속 송치했다고 밝혔다.
A씨는 교통사고처리법상 12대 중과실에 해당되는 '중앙선 침범'으로 역주행 교통사고를 일으킨 혐의를 받는다.
경찰에 따르면 스포츠유틸리티차량(SUV) 운전자 A씨는 5월 19일 밤 11시 50분쯤 부산진구 가야고가교 밑 도로에서 중앙선을 넘어 역주행으로 140m를 달려
반대편 차선의 오토바이 운전자 조모(16)군을 들이받았다. 조군은 원동기장치자전거 면허를 취득한 상태였고 헬멧도 쓰고 있었지만 크게 다쳤다.
사고 당일 수술을 받았으나 얼마 후 2차 뇌출혈로 뇌사 판정이 내려졌고, 사고 발생 약 한 달 만인 지난달 16일 끝내 사망했다.
사고를 낸 A씨는 술을 마시거나 약물을 복용한 상태에서 운전하지는 않은 것으로 조사됐다.
경찰 관계자는 'A씨가 자신이 정주행을 하고 오토바이가 역주행을 한 것으로 착각했다고 진술했다'고 설명했다.
model
한국어 요약:
model
부산의 한 왕복 2차선 도로에서 역주행 사고로 배달 오토바이 운전자인 고등학생이 숨지는 사고가 발생했다. 유족은 '가해자가 사고 후 곧바로 신고하지 않고 늑장 대응해 피해를 키웠다'고 주장하고 있다. 경찰은 교통사고처리법상 업무상 과실치사 혐의로 지난 3일 A(59)씨를 검찰에 불구속 송치했다.


### 위에서 본 키워드 추출 기능과 요약기능을 한 번에 다 보여주는 프롬프트 작성

In [9]:
# Input_text : 위 정의된 기사와 동일 

def change_inference_chat_format(input_text):
    return [
    {"role": "user", "content": f"다음 텍스트를 한국어로 간단히 요약하고, 관련된 5개의 키워를 추출해주세요:\n{input_text}"},
	{"role": "assistant", "content": ""},
    ]
prompt = change_inference_chat_format(input_text)
# tokenizer 초기화 및 적용t\
inputs = tokenizer.apply_chat_template(prompt, tokenize=True, add_generation_prompt=True, return_tensors="pt").to(model.device)
outputs = model.generate(
    inputs,
    max_new_tokens=256,
    use_cache=True
)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

user
다음 텍스트를 한국어로 간단히 요약하고, 관련된 5개의 키워를 추출해주세요:
다음 텍스트를 한국어로 간단히 요약해주세요:
부산의 한 왕복 2차선 도로에서 역주행 사고로 배달 오토바이 운전자인 고등학생이 숨지는 사고가 발생했다.
유족은 '가해자가 사고 후 곧바로 신고하지 않고 늑장 대응해 피해를 키웠다'고 주장하고 있다.
11일 부산진경찰서는 교통사고처리특례법(교통사고처리법)상 업무상 과실치사 혐의로 지난 3일 A(59)씨를 검찰에 불구속 송치했다고 밝혔다.
A씨는 교통사고처리법상 12대 중과실에 해당되는 '중앙선 침범'으로 역주행 교통사고를 일으킨 혐의를 받는다.
경찰에 따르면 스포츠유틸리티차량(SUV) 운전자 A씨는 5월 19일 밤 11시 50분쯤 부산진구 가야고가교 밑 도로에서 중앙선을 넘어 역주행으로 140m를 달려
반대편 차선의 오토바이 운전자 조모(16)군을 들이받았다. 조군은 원동기장치자전거 면허를 취득한 상태였고 헬멧도 쓰고 있었지만 크게 다쳤다.
사고 당일 수술을 받았으나 얼마 후 2차 뇌출혈로 뇌사 판정이 내려졌고, 사고 발생 약 한 달 만인 지난달 16일 끝내 사망했다.
사고를 낸 A씨는 술을 마시거나 약물을 복용한 상태에서 운전하지는 않은 것으로 조사됐다.
경찰 관계자는 'A씨가 자신이 정주행을 하고 오토바이가 역주행을 한 것으로 착각했다고 진술했다'고 설명했다.
model

model
* 부산의 한 왕복 2차선 도로에서 역주행 사고로 배달 오토바이 운전자인 고등학생이 숨지는 사고가 발생했다.
* 유족은 '가해자가 사고 후 곧바로 신고하지 않고 늑장 대응해 피해를 키웠다'고 주장하고 있다.
* A씨는 교통사고처리법상 12대 중과실에 해당되는 '중앙선 침범'으로 역주행 교통사고를 일으킨 혐의를 받는다.
* 사고를 낸 A씨는 술을 마시거나 약물을 복용한 상태에서 운전하지는 않은 것으로 조사됐다.
* 경찰 관계자는 'A씨가 자신이 정주행을 하고 오토바이가 역주행을 한 것으로 착각했다고 진술했다.


→ 원하는대로 나오지 않음. 사용자가 원하는 것은 주어진 글의 핵심을 한 문장으로 간단히 요약하고, 기사에서 5개의 핵심 단어를 뽑아 내는 것

### 3.4.5. 키워드 데이터 생성

가져온 데이터셋은 각 기사별 키워드 정보를 포함하지 않음.   
따라서 Gemma 모델을 활용해 각 데이터마다 5개의씩 키워드를 추출함.

In [ ]:
from transformers import pipeline 

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, device_map="auto", batch_size=16)
def key_word_prompt(input_text, summary_text):
    return [
    {"role": "user", "content": f"{input_text}"},
    {"role": "assistant", "content": f"{summary_text}"},
    {"role": "user", "content": "중요한 키워드 5개를 뽑아주세요."},
    {"role": "assistant", "content": ""}
    ]

# 데이터셋 batch 단위로 키워드를 생성
def extract_keywords_batch(batch):
    # 각 데이터에 대해 프롬프트를 생성함 (text1, sum1) -> prompt1
    prompts = [key_word_prompt(original, summary) for original, summary in zip(batch["original"], batch["summary"])]

    generated_texts = pipe(prompts, max_new_tokens=150, return_full_text=False)
    keywords = [gen_text[0]["generated_text"] for gen_text in generated_texts]
    batch["keywords"] = keywords
    return batch


The model 'PeftModelForCausalLM' is not supported for text-generation. Supported models are ['BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'ElectraForCausalLM', 'ErnieForCausalLM', 'FalconForCausalLM', 'FuyuForCausalLM', 'GemmaForCausalLM', 'Gemma2ForCausalLM', 'GitForCausalLM', 'GPT2LMHeadModel', 'GPT2LMHeadModel', 'GPTBigCodeForCausalLM', 'GPTNeoForCausalLM', 'GPTNeoXForCausalLM', 'GPTNeoXJapaneseForCausalLM', 'GPTJForCausalLM', 'JambaForCausalLM', 'JetMoeForCausalLM', 'LlamaForCausalLM', 'MambaForCausalLM', 'Mamba2ForCausalLM', 'MarianForCausalLM', 'MBartForCausalLM', 'MegaForCausalLM', 'MegatronBertForCausalLM', 'MistralForCausalLM', 'MixtralForCausal

In [11]:
import torch
print(torch.__version__)            # 2.x.x+cu124
print(torch.cuda.is_available())    # True
print(torch.cuda.get_device_name(0))  # NVIDIA GeForce RTX 2080 Ti
print(torch.cuda.device_count())    # 2
print(model.device)  # cuda:0 이어야 함

2.4.1+cu124
True
NVIDIA GeForce RTX 2080 Ti
2
cuda:0


In [12]:
# dataset에 keyword 열 추가 (batch 단위로 처리) 
#sample_dataset = dataset["train"].shuffle(seed=42).select(range(1000))
#sample_dataset = sample_dataset.map(extract_keywords_batch, batched=True, batch_size=16)  # 적절한 batch_size 선택

# sample_dataset

sample_dataset = datasets.load_dataset("daje/keyword_summary")["train"]
print(sample_dataset[0])  # keywords 컬럼 있는지 확인

{'original': '전두환 정권을 흔들었던 \'큰 손\' 장영자의 네 번째 유죄가 확정됐다.\n  혐의는 사기, 징역은 4년이다.\n  대법원 2부(주심 박상옥 대법관)는 9일 무죄를 주장한 장씨의 상고를 기각했다.\n  장씨는 지인들에게서 세 차례에 걸쳐 6억여원을 가로채고 이들에게 위조 수표로 수억원을 요구한 혐의를 받는다.\n  올해로 76세를 맞은 장씨는 이미 복역을 마친 29년을 포함해 인생의 절반가량인 33년을 감옥에서 보내게됐다.\n   우수한 인재에게만 허락된다는 숙명여대 \'5월의 여왕(메이퀸)\' 출신인 장영자의 노년이 씁쓸하다.\n  장씨는 전두환 전 대통령의 처삼촌인 고(故) 이규광씨의 처제이자, 중앙정보부 차장이었던 고(故) 이철희씨의 아내였다.\n  전두환 정권 흔든 장영자의 삶장씨의 앞선 범행과 비교해 네 번째 사기 액수는 그 규모가 상당히 줄어든 편이다.\n  장씨는 1980년대 전두환 전 대통령과 남편을 내세워 자금 압박에 시달리던 기업에 자금을 빌려준 뒤 몇 배에 달하는 어음을 할인 유통하며 이득을 챙겼다.\n  그 규모가 총 7111억원에 달했다.\n  장씨는 이중 6404억원의 어음을 할인해 사용했다.\n   실체가 없던 어음은 부도가 났고 기업들은 도산했다.\n  장씨는 이 첫 번째 사기로 15년형을 선고받았다.\n  10년만인 1992년에 가석방됐다.\n  전두환 전 대통령의 부인 이순자씨는 자서전 『당신은 외롭지 않다』에서 이를 "조금씩 민심도 안정되고 경제도 생기를 되찾아 (남편이) 자신감을 얻던 시점에 날벼락같이 찾아온 횡액과도 같은 사건"이라 회고했다.\n  장씨가 당시 법정에서 아직 시중에 유통중인 어음이 있다며 "경제는 유통""나는 권력투쟁의 희생양이라"이라 했던 말은 유행어가 됐다.\n  \'장영자 어음할인 사건\' 수사 축소 의혹으로 두 명의 법무부 장관이 경질됐다.\n   "누나 성질 급하다" 70대 장영자의 돈 요구장씨는 출소한지 2년만인 1994년에 140억원대 차용 사기를 저질렀다.\n  두 번

In [13]:
sample_dataset[0]

{'original': '전두환 정권을 흔들었던 \'큰 손\' 장영자의 네 번째 유죄가 확정됐다.\n  혐의는 사기, 징역은 4년이다.\n  대법원 2부(주심 박상옥 대법관)는 9일 무죄를 주장한 장씨의 상고를 기각했다.\n  장씨는 지인들에게서 세 차례에 걸쳐 6억여원을 가로채고 이들에게 위조 수표로 수억원을 요구한 혐의를 받는다.\n  올해로 76세를 맞은 장씨는 이미 복역을 마친 29년을 포함해 인생의 절반가량인 33년을 감옥에서 보내게됐다.\n   우수한 인재에게만 허락된다는 숙명여대 \'5월의 여왕(메이퀸)\' 출신인 장영자의 노년이 씁쓸하다.\n  장씨는 전두환 전 대통령의 처삼촌인 고(故) 이규광씨의 처제이자, 중앙정보부 차장이었던 고(故) 이철희씨의 아내였다.\n  전두환 정권 흔든 장영자의 삶장씨의 앞선 범행과 비교해 네 번째 사기 액수는 그 규모가 상당히 줄어든 편이다.\n  장씨는 1980년대 전두환 전 대통령과 남편을 내세워 자금 압박에 시달리던 기업에 자금을 빌려준 뒤 몇 배에 달하는 어음을 할인 유통하며 이득을 챙겼다.\n  그 규모가 총 7111억원에 달했다.\n  장씨는 이중 6404억원의 어음을 할인해 사용했다.\n   실체가 없던 어음은 부도가 났고 기업들은 도산했다.\n  장씨는 이 첫 번째 사기로 15년형을 선고받았다.\n  10년만인 1992년에 가석방됐다.\n  전두환 전 대통령의 부인 이순자씨는 자서전 『당신은 외롭지 않다』에서 이를 "조금씩 민심도 안정되고 경제도 생기를 되찾아 (남편이) 자신감을 얻던 시점에 날벼락같이 찾아온 횡액과도 같은 사건"이라 회고했다.\n  장씨가 당시 법정에서 아직 시중에 유통중인 어음이 있다며 "경제는 유통""나는 권력투쟁의 희생양이라"이라 했던 말은 유행어가 됐다.\n  \'장영자 어음할인 사건\' 수사 축소 의혹으로 두 명의 법무부 장관이 경질됐다.\n   "누나 성질 급하다" 70대 장영자의 돈 요구장씨는 출소한지 2년만인 1994년에 140억원대 차용 사기를 저질렀다.\n  두 번

### 3.4.6 데이터 전처리

Gemma 모델이 쉽게 이햏라 수 있도록 대화 형식으로 데이터를 전처리

In [ ]:
# 해당 함수는 주어진 텍스트 데이터를 사용자와 모델 간의 대화 형식으로 바꾸는 역할을 함. 
# 이는 텍스트를 요약하고 관련 키워드를 뽑아내는 대화를 만듬.
def chat_keyword_summary_format(example):

        return [
            {"role": "user", "content": f"다음 텍스트를 한국어로 간단히 요약 및 관련 키워를 추출해주세요:\n{example['original']}"},
            {"role": "assistant", "content": f"한국어 요약:{example['summary']}\n키워드:{example['keywords']}"}
        ]

print(sample_dataset[0])

formatted = tokenizer.apply_chat_template(
    chat_keyword_summary_format(sample_dataset[0]), tokenize=False
)
print(formatted)

{'original': '전두환 정권을 흔들었던 \'큰 손\' 장영자의 네 번째 유죄가 확정됐다.\n  혐의는 사기, 징역은 4년이다.\n  대법원 2부(주심 박상옥 대법관)는 9일 무죄를 주장한 장씨의 상고를 기각했다.\n  장씨는 지인들에게서 세 차례에 걸쳐 6억여원을 가로채고 이들에게 위조 수표로 수억원을 요구한 혐의를 받는다.\n  올해로 76세를 맞은 장씨는 이미 복역을 마친 29년을 포함해 인생의 절반가량인 33년을 감옥에서 보내게됐다.\n   우수한 인재에게만 허락된다는 숙명여대 \'5월의 여왕(메이퀸)\' 출신인 장영자의 노년이 씁쓸하다.\n  장씨는 전두환 전 대통령의 처삼촌인 고(故) 이규광씨의 처제이자, 중앙정보부 차장이었던 고(故) 이철희씨의 아내였다.\n  전두환 정권 흔든 장영자의 삶장씨의 앞선 범행과 비교해 네 번째 사기 액수는 그 규모가 상당히 줄어든 편이다.\n  장씨는 1980년대 전두환 전 대통령과 남편을 내세워 자금 압박에 시달리던 기업에 자금을 빌려준 뒤 몇 배에 달하는 어음을 할인 유통하며 이득을 챙겼다.\n  그 규모가 총 7111억원에 달했다.\n  장씨는 이중 6404억원의 어음을 할인해 사용했다.\n   실체가 없던 어음은 부도가 났고 기업들은 도산했다.\n  장씨는 이 첫 번째 사기로 15년형을 선고받았다.\n  10년만인 1992년에 가석방됐다.\n  전두환 전 대통령의 부인 이순자씨는 자서전 『당신은 외롭지 않다』에서 이를 "조금씩 민심도 안정되고 경제도 생기를 되찾아 (남편이) 자신감을 얻던 시점에 날벼락같이 찾아온 횡액과도 같은 사건"이라 회고했다.\n  장씨가 당시 법정에서 아직 시중에 유통중인 어음이 있다며 "경제는 유통""나는 권력투쟁의 희생양이라"이라 했던 말은 유행어가 됐다.\n  \'장영자 어음할인 사건\' 수사 축소 의혹으로 두 명의 법무부 장관이 경질됐다.\n   "누나 성질 급하다" 70대 장영자의 돈 요구장씨는 출소한지 2년만인 1994년에 140억원대 차용 사기를 저질렀다.\n  두 번

이제 설명한 apply_chat_template을 적용하고 토크나이저를 이용해 문자를 숫자로 변환하는 작업을 적용

In [ ]:
EOS_TOKEN = tokenizer.eos_token
def tokenize(element):
    formatted = tokenizer.apply_chat_template(
        chat_keyword_summary_format(element), tokenize=False
    ) + EOS_TOKEN

    outputs = tokenizer(formatted)
    
    return {
        "input_ids": outputs["input_ids"],
        "attention_mask": outputs["attention_mask"],
    }

tokenized_sample_dataset = sample_dataset.map(tokenize)

### 3.4.7. 데이터셋 분리 및 콜레이터 설정

In [ ]:
# 데이터셋 분할 코드
# 훈련용과 테스트용으로 데이터를 나누는 함수
tokenized_sample_dataset = tokenized_sample_dataset.train_test_split(
    test_size=0.1, 
    seed=42 # 데이터 분할 시 랜덤성을 고정하는 값
)
tokenized_sample_dataset

DatasetDict({
    train: Dataset({
        features: ['original', 'summary', 'keywords', 'input_ids', 'attention_mask'],
        num_rows: 900
    })
    test: Dataset({
        features: ['original', 'summary', 'keywords', 'input_ids', 'attention_mask'],
        num_rows: 100
    })
})

In [ ]:
# 데이터를 직접 만드는데, 1시간 이상 소요되므로 미리 만들어놓았습니다. 
# 필요하신 분들은 주석을 풀고 사용해주세요 
# 위에 반환된 결과가 이를 실행하면 나옴

# sample_dataset = datasets.load_dataset("daje/keyword_summary")
# print(sample_dataset["train"][0])
# tokenized_sample_dataset = sample_dataset.map(tokenize)
# tokenized_sample_dataset = tokenized_sample_dataset['train'].train_test_split(test_size=0.1, seed=42)
# tokenized_sample_dataset


콜레이터 설정
- 콜레이터의 주요 기능은 데이터 배치화, 패딩(데이터의 길이 맞춤), 텐서 변화(데이터 타입 변환)이 있으며,  
중요하게는 **라벨링 작업**도 담장함

In [ ]:
# 실습은 콜레이터를 주로 라벨러 역할로 함
response_template_ids = tokenizer.encode(
    "<start_of_turn>model\n", 
    add_special_tokens=False
    )
    
collator = DataCollatorForCompletionOnlyLM(
    response_template_ids, tokenizer=tokenizer
)

### 3.4.8 학습 파라미터 설정 

In [ ]:
# wandb 사용
# wandb는 머신러닝 실험을 추적하고 시각화하는 데 널리 쓰이는 플랫폼
import wandb
wandb.login(key="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")


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\mobile\_netrc
wandb: Currently logged in as: cook100873 (cook100873-hanbat-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
# 학습에 사용되는 학습 파라미터를 우선 설정
import wandb
from transformers import TrainingArguments

wandb.init(project="gemma-2B-it-Full-Fine-Tuning", entity="cook100873-hanbat-university")

# HuggingFace Trainer가 학습할 때 사용하는 모든 설정을 담는 클래스
training_args = TrainingArguments(
    # 학습 중 생성되는 모델 체크포인트와 결과를 저장할 디렉토리
    output_dir="./keywords_gemma_results",
    # 전체 학습을 진행할 최대 스텝 수
    max_steps=800,
    # GPU(또는 장치) 하나당 학습 시 사용할 배치 크기
    per_device_train_batch_size=1,
    # GPU(또는 장치) 하나당 평가 시 사용할 배치 크기
    per_device_eval_batch_size=1,
    # 여러 step의 gradient를 누적해 실제 배치 크기를 늘리는 설정
    gradient_accumulation_steps=4,
    # 중간 activation을 일부만 저장하여 GPU 메모리를 절약하는 설정
    gradient_checkpointing=True,
    # bfloat16 정밀도를 사용하여 연산 속도와 메모리 효율을 높이는 설정
    bf16=True,                          # ← 추가: bfloat16 활성화
    # 학습에 사용할 optimizer 방식 (여기서는 paged_adamw_8bit)
    optim="paged_adamw_8bit",
    # 학습 초기에 learning rate를 점진적으로 증가시키는 step 수
    warmup_steps=0,
    # 과적합 방지를 위한 가중치 감소(regularization) 값
    weight_decay=0.01,
    # 모델 파라미터를 업데이트할 때 사용하는 학습률
    learning_rate=2e-4,
    # 학습 로그를 저장할 디렉토리
    logging_dir="./logs",
    # 몇 step마다 로그를 기록할지 설정
    logging_steps=100,
    # 학습 중 평가를 수행하는 방식 (여기서는 비활성화)
    evaluation_strategy="no",           # ← 추가: eval 비활성화로 VRAM 절약
    # 모델 체크포인트를 저장하는 방식
    save_strategy="steps",
    # 몇 step마다 모델 체크포인트를 저장할지 설정
    save_steps=200,
    # 학습 로그를 기록할 플랫폼 (여기서는 WandB)
    report_to="wandb",
    # 데이터 로딩 시 메모리 고정 여부 설정 (메모리 안정성 관련 옵션)
    dataloader_pin_memory=False,        # ← 추가
)

c:\Users\mobile\AppData\Local\mise\installs\python\3.12.13\Lib\site-packages\transformers\training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


### 3.4.9 평가 메트릭 정의 

평가 메트릭은 모델이 얼마나 잘 학습됐는지 측정하기 위한 지표. 
-> 이번 실습에서는 기계 번역 품질을 측정하는 BLEU 점수와 모델 예측 결과가 실제 정답과 얼마나 정확하게 일치하는 지를 측정하는 정확(ACC)를 평가 지표로 사용함. 
  
BLEU는 생성된 텍스트와 참조 텍스트 사이의 역속된 n개 단어(n-gram)의 일치도를 바탕으로 평가를 진행하기에 요약과 키워드 추출에도 응용해 사용할 수 있음.  
→ 이 과정에서 생성된 텍스트에서 사용된 n-gram이 참조 텍스트에도 얼마나 존재하는지를 나타내는 비율인 정밀도와 텍스트 길이의 간결성 패널티를 고려함. 

→ BLEU 점수는 최종적으로 0에서 1사이의 값으로 나타냄. 

→ 이때 1에 가까울수록 생성된 텍스트가 참조 텍스트와 더 비슷함. 

In [21]:
import evaluate

bleu = evaluate.load("bleu")
acc = evaluate.load("accuracy")

def preprocess_logits_for_metrics(logits, labels):
    if isinstance(logits, tuple):
        # 모델과 설정에 따라 logits에는 추가적인 텐서들이 포함될 수 있습니다.
        # 예를 들어, past_key_values 같은 것들이 있을 수 있지만,
        # logits는 항상 첫 번째 요소입니다.
        logits = logits[0]
    # 토큰 ID를 얻기 위해 argmax를 수행합니다.
    return logits.argmax(dim=-1)

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    # preds는 labels와 같은 형태를 갖습니다.
    # preprocess_logits_for_metrics에서 argmax(-1)가 계산된 후입니다.
    # 하지만 우리는 labels를 한 칸 이동시켜야 합니다.
    labels = labels[:, 1:]
    preds = preds[:, :-1]

    # -100은 DataCollatorForCompletionOnlyLM에서 사용되는 
    # ignore_index의 기본값입니다.
    mask = labels == -100
    # -100을 토크나이저가 디코드할 수 있는 값으로 대체합니다.
    labels[mask] = tokenizer.pad_token_id
    preds[mask] = tokenizer.pad_token_id

    # BLEU 점수는 텍스트를 입력으로 받기 때문에,
    # 토큰 ID에서 텍스트로 변환해야 합니다.
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    bleu_score = bleu.compute(predictions=decoded_preds, references=decoded_labels)
    
    # accuracy는 정수 리스트를 입력으로 받습니다.
    # 우리는 -100이 아닌 부분만 평가하고 싶기 때문에,
    # 마스크의 부정(~)을 사용합니다.
    accuracy = acc.compute(predictions=preds[~mask], references=labels[~mask])

    return {**bleu_score, **accuracy}

- preprocess_logits_for_metrics함수는 모델의 출력을 가장 높은 확률의 토큰 ID로 바꿈
- compute_metrics 함수는 실제 평가를 수행하며, 예측값과 실제값을 비교함

이 과정에서 패딩이나 무시해야 할 토큰을 나타내는 특별한 값(-100)을 제외하고 유효한 토큰만을 평가함. 토큰 ID를 텍스트로 바꾼호, BLUE(Bilingual Evaluation Understudy) 점수를 계산함. 

**→ BLEU는 텍스트가 참조 텍스트와 얼마나 비슷한지를 평가함**

→ 또한 정확도도 계산하되 유효한 토큰에 대해서만 평가함

### 3.4.10 모델 학습 및 평가

Trainer 클래스는 모델 학습에 필요한 다양한 설정을 한 곳에서 관리하고, 학습 과정을 자동화할 수 있음

In [22]:
from transformers import Trainer, TrainingArguments
from transformers.integrations import WandbCallback

trainer = Trainer(
    args=training_args,
    model=model,
    tokenizer=tokenizer,
    data_collator=collator,
    train_dataset=tokenized_sample_dataset["train"],
    eval_dataset=tokenized_sample_dataset["test"],
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,
    compute_metrics=compute_metrics,
    # callbacks=[WandbCallback()]
)

max_steps is given, it will override any value given in num_train_epochs


In [ ]:
# trainer.train()은 이전에 설정한 Trainer 객체를 활용해 모델의 학습 과정을 시작함. 
trainer.train()
trainer.save_model("./gemma-lora-finetuned")  

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


  0%|          | 0/800 [00:00<?, ?it/s]

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
c:\Users\mobile\AppData\Local\mise\installs\python\3.12.13\Lib\site-packages\torch\_dynamo\eval_frame.py:600: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
c:\Users\mobile\AppData\Local\mise\installs\python\3.12.13\Lib\site-packages\torch\utils\checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]


{'loss': 0.5645, 'grad_norm': 1.5239636898040771, 'learning_rate': 0.000175, 'epoch': 0.44}
{'loss': 0.3853, 'grad_norm': 1.099950909614563, 'learning_rate': 0.00015000000000000001, 'epoch': 0.89}


c:\Users\mobile\AppData\Local\mise\installs\python\3.12.13\Lib\site-packages\torch\_dynamo\eval_frame.py:600: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
c:\Users\mobile\AppData\Local\mise\installs\python\3.12.13\Lib\site-packages\torch\utils\checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]


{'loss': 0.3188, 'grad_norm': 1.2636750936508179, 'learning_rate': 0.000125, 'epoch': 1.33}
{'loss': 0.2898, 'grad_norm': 1.2701010704040527, 'learning_rate': 0.0001, 'epoch': 1.78}


c:\Users\mobile\AppData\Local\mise\installs\python\3.12.13\Lib\site-packages\torch\_dynamo\eval_frame.py:600: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
c:\Users\mobile\AppData\Local\mise\installs\python\3.12.13\Lib\site-packages\torch\utils\checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]


{'loss': 0.2789, 'grad_norm': 1.3024377822875977, 'learning_rate': 7.500000000000001e-05, 'epoch': 2.22}
{'loss': 0.2152, 'grad_norm': 2.1269915103912354, 'learning_rate': 5e-05, 'epoch': 2.67}


c:\Users\mobile\AppData\Local\mise\installs\python\3.12.13\Lib\site-packages\torch\_dynamo\eval_frame.py:600: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
c:\Users\mobile\AppData\Local\mise\installs\python\3.12.13\Lib\site-packages\torch\utils\checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]


{'loss': 0.1987, 'grad_norm': 1.0518230199813843, 'learning_rate': 2.5e-05, 'epoch': 3.11}
{'loss': 0.1727, 'grad_norm': 1.132186770439148, 'learning_rate': 0.0, 'epoch': 3.56}
{'train_runtime': 8321.2862, 'train_samples_per_second': 0.385, 'train_steps_per_second': 0.096, 'train_loss': 0.30299007415771484, 'epoch': 3.56}


In [ ]:
import torch
from transformers import Trainer

torch.cuda.empty_cache()

trainer = Trainer(
    args=training_args,
    model=model,
    tokenizer=tokenizer,
    data_collator=collator,
    train_dataset=tokenized_sample_dataset["train"],
    eval_dataset=tokenized_sample_dataset["test"],
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,
    compute_metrics=compute_metrics,
)

# 이는 Trainer 객체를 사용해 모델의 성능을 평가함
trainer.evaluate()

max_steps is given, it will override any value given in num_train_epochs


  0%|          | 0/100 [00:00<?, ?it/s]

{'eval_loss': 0.43719279766082764,
 'eval_model_preparation_time': 0.0,
 'eval_bleu': 0.6612139279160867,
 'eval_precisions': [0.8195876288659794,
  0.7082880926529135,
  0.622558053814965,
  0.5484416072099136],
 'eval_brevity_penalty': 0.9909759086377132,
 'eval_length_ratio': 0.9910163818918443,
 'eval_translation_length': 5626,
 'eval_reference_length': 5677,
 'eval_accuracy': 0.9142404845481451,
 'eval_runtime': 88.236,
 'eval_samples_per_second': 1.133,
 'eval_steps_per_second': 1.133}

-> 보면 eval_loss는 0.43임 이 값이 작을 수록 좋음.  
-> eval_bleu 점수는 0.661으로, 이는 모델이 만든 텍스트의 질을 나타내며 1에 가까울 수록 좋음  
-> eval_accuracy는 0.914로, 이는 모델의 정확도를 보여주며 약 90%의 정확성을 의미함  

#### 3.4.11 파인튜닝한 모델 테스트

In [25]:
input_text = "부산의 한 왕복 2차선 도로에서 역주행 사고로 배달 오토바이 운전자인 고등학생이 숨지는 사고가 발생했다. 유족은 '가해자가 사고 후 곧바로 신고하지 않고 늑장 대응해 피해를 키웠다'고 주장하고 있다.\n11일 부산진경찰서는 교통사고처리특례법(교통사고처리법)상 업무상 과실치사 혐의로 지난 3일 A(59)씨를 검찰에 불구속 송치했다고 밝혔다. A씨는 교통사고처리법상 12대 중과실에 해당되는 '중앙선 침범'으로 역주행 교통사고를 일으킨 혐의를 받는다.\n경찰에 따르면 스포츠유틸리티차량(SUV) 운전자 A씨는 5월 19일 밤 11시 50분쯤 부산진구 가야고가교 밑 도로에서 중앙선을 넘어 역주행으로 140m를 달려 반대편 차선의 오토바이 운전자 조모(16)군을 들이받았다. 조군은 원동기장치자전거 면허를 취득한 상태였고 헬멧도 쓰고 있었지만 크게 다쳤다. 사고 당일 수술을 받았으나 얼마 후 2차 뇌출혈로 뇌사 판정이 내려졌고, 사고 발생 약 한 달 만인 지난달 16일 끝내 사망했다. 사고를 낸 A씨는 술을 마시거나 약물을 복용한 상태에서 운전하지는 않은 것으로 조사됐다. 경찰 관계자는 'A씨가 자신이 정주행을 하고 오토바이가 역주행을 한 것으로 착각했다고 진술했다'고 설명했다."

def get_chat_format(input_text):

        return [
            {"role": "user", "content": f"다음 텍스트를 한국어로 간단히 요약 및 관련 키워드를 추출해주세요:\n{input_text}"},
            {"role": "assistant", "content": "한국어 요약:\n키워드:"}
        ]


def change_inference_chat_format(input_text):
    return [
    {"role": "user", "content": f"{input_text}"},
    {"role": "assistant", "content": ""}
    ]
prompt = change_inference_chat_format(input_text)
# tokenizer 초기화 및 적용t\
inputs = tokenizer.apply_chat_template(prompt, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
outputs = model.generate(
    inputs,
    max_new_tokens=512,
    use_cache=True
)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

user
부산의 한 왕복 2차선 도로에서 역주행 사고로 배달 오토바이 운전자인 고등학생이 숨지는 사고가 발생했다. 유족은 '가해자가 사고 후 곧바로 신고하지 않고 늑장 대응해 피해를 키웠다'고 주장하고 있다.
11일 부산진경찰서는 교통사고처리특례법(교통사고처리법)상 업무상 과실치사 혐의로 지난 3일 A(59)씨를 검찰에 불구속 송치했다고 밝혔다. A씨는 교통사고처리법상 12대 중과실에 해당되는 '중앙선 침범'으로 역주행 교통사고를 일으킨 혐의를 받는다.
경찰에 따르면 스포츠유틸리티차량(SUV) 운전자 A씨는 5월 19일 밤 11시 50분쯤 부산진구 가야고가교 밑 도로에서 중앙선을 넘어 역주행으로 140m를 달려 반대편 차선의 오토바이 운전자 조모(16)군을 들이받았다. 조군은 원동기장치자전거 면허를 취득한 상태였고 헬멧도 쓰고 있었지만 크게 다쳤다. 사고 당일 수술을 받았으나 얼마 후 2차 뇌출혈로 뇌사 판정이 내려졌고, 사고 발생 약 한 달 만인 지난달 16일 끝내 사망했다. 사고를 낸 A씨는 술을 마시거나 약물을 복용한 상태에서 운전하지는 않은 것으로 조사됐다. 경찰 관계자는 'A씨가 자신이 정주행을 하고 오토바이가 역주행을 한 것으로 착각했다고 진술했다'고 설명했다.
model

model
부산의 한 왕복 2차선 도로에서 역주행 사고로 배달 오토바이 운전자인 고등학생이 숨지는 사고가 발생했다. 유족은 '가해자가 사고 후 곧바로 신고하지 않고 늑장 대응해 피해를 키웠다'고 주장하고 있다.

11일 부산진경찰서는 교통사고처리특례법(교통사고처리법)상 업무상 과실치사 혐의로 지난 3일 A(59)씨를 검찰에 불구속 송치했다고 밝혔다. A씨는 교통사고처리법상 12대 중과실에 해당되는 '중앙선 침범'으로 역주행 교통사고를 일으킨 혐의를 받는다.

경찰에 따르면 스포츠유틸리티차량(SUV) 운전자 A씨는 5월 19일 밤 11시 50분쯤 부산진구 가야고가교 밑 도로에서 중앙선을 넘어 역주행으로 140m를 달려 반대편 차선의 오토바이 운전자 조모(16)군을 들이받았다. 조군